# Lab 1 — First Steps into Agentic AI

My first practical lab working with LLM APIs and multi-step AI workflows.

**Focus:** API setup, prompting, chaining LLM calls, and evaluating responses.

*Based on concepts from Ed Donner's Agentic AI course, with my own adaptations and experiments.*


## 1. Setting up the API environment

I’m using environment variables to keep API credentials separate from the notebook code.


In [2]:
# Load the environment-variable helper.


from dotenv import load_dotenv


In [3]:
# Load the variables from .env so the API credentials are available to Python.


load_dotenv(override=True)



True

In [4]:
# Quick check that the notebook is running.

print("TEST")


TEST


The API key is loaded from `.env` rather than written directly into the notebook.

Only a short prefix is displayed when checking the key, so the full credential is not exposed.


In [5]:

import os
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print("OpenAI API Key is available.")
else:
    print("OpenAI API Key not set.")
    


OpenAI API Key is available.


## 2. Making the first LLM call

I’m using the OpenAI client to send a simple agriculture-related prompt and inspect the response.


In [6]:
# Import the OpenAI client used for the API calls in this lab.


from openai import OpenAI


In [7]:
# Create a client using the API credentials loaded from the environment.


openai = OpenAI()


In [8]:
# Start with a simple agriculture-focused prompt.


messages = [{"role": "user", "content": "tell me a fun fact in agriculture"}]


In [9]:
messages

[{'role': 'user', 'content': 'tell me a fun fact in agriculture'}]

In [10]:
# Send the prompt to the model and display its response.


response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)


Fun fact: **Worms are “mini plows” for soil**—as they burrow, they mix organic matter deeper into the ground and create channels that help **water and air reach plant roots**.


## 3. Chaining LLM calls

Next, I use one LLM call to generate a challenging question, then use another call to answer it.

This is where the workflow starts becoming more interesting: the output from one step becomes the input to the next.


In [11]:
# Ask the model to generate a challenging question.


question = "Please propose a hard, challenging question to assess someone's IQ. Respond only with the question."
messages = [{"role": "user", "content": question}]


In [12]:
# Generate the question with the selected model. "gpt-5.4-mini"


response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
question = response.choices[0].message.content
print(question)


You have 12 identical-looking coins, one of which is counterfeit and may be either heavier or lighter than the others. Using a balance scale exactly three times, what is a strategy that will always identify the counterfeit coin and determine whether it is heavier or lighter?


In [13]:
# Pass the generated question into the next step.


messages = [{"role": "user", "content": question}]
messages


[{'role': 'user',
  'content': 'You have 12 identical-looking coins, one of which is counterfeit and may be either heavier or lighter than the others. Using a balance scale exactly three times, what is a strategy that will always identify the counterfeit coin and determine whether it is heavier or lighter?'}]

In [14]:
# Ask the model to answer the generated question.


response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
answer = response.choices[0].message.content
print(answer)


Here’s a standard 3-weighing strategy that always works for **12 coins**, with **one counterfeit coin** that could be **heavier or lighter**.

I’ll label the coins:

**1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12**

---

## Weighing 1
Weigh:

**1 2 3 4** vs **5 6 7 8**

There are three possible outcomes.

---

# Case 1: The first weighing balances
Then coins **1–8 are genuine**, so the counterfeit is among **9, 10, 11, 12**.

## Weighing 2
Weigh:

**9 10 11** vs **1 2 3**

Since 1, 2, 3 are genuine:

- If left side is heavier, then one of **9, 10, 11** is **heavy**
- If left side is lighter, then one of **9, 10, 11** is **light**
- If it balances, then **12** is counterfeit

## Weighing 3
- If 9, 10, 11 are suspect, weigh **9 vs 10**
  - If one is heavier/lighter as expected, you know which one
  - If they balance, then **11** is the counterfeit
- If 12 was the suspect, weigh **12 vs 1**
  - If 12 is heavier or lighter, you know which

That handles the balance case.

---

# Case 2: The first 

In [15]:
from IPython.display import Markdown, display

display(Markdown(answer))

Here’s a standard 3-weighing strategy that always works for **12 coins**, with **one counterfeit coin** that could be **heavier or lighter**.

I’ll label the coins:

**1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12**

---

## Weighing 1
Weigh:

**1 2 3 4** vs **5 6 7 8**

There are three possible outcomes.

---

# Case 1: The first weighing balances
Then coins **1–8 are genuine**, so the counterfeit is among **9, 10, 11, 12**.

## Weighing 2
Weigh:

**9 10 11** vs **1 2 3**

Since 1, 2, 3 are genuine:

- If left side is heavier, then one of **9, 10, 11** is **heavy**
- If left side is lighter, then one of **9, 10, 11** is **light**
- If it balances, then **12** is counterfeit

## Weighing 3
- If 9, 10, 11 are suspect, weigh **9 vs 10**
  - If one is heavier/lighter as expected, you know which one
  - If they balance, then **11** is the counterfeit
- If 12 was the suspect, weigh **12 vs 1**
  - If 12 is heavier or lighter, you know which

That handles the balance case.

---

# Case 2: The first weighing does not balance
Assume first:

**1 2 3 4** vs **5 6 7 8**

Then the counterfeit is among those 8 coins, and we also know whether it is in the heavy side or the light side.

Without loss of generality, suppose:

- **1 2 3 4 is heavier** than **5 6 7 8**

Then either:
- one of **1,2,3,4** is **heavy**, or
- one of **5,6,7,8** is **light**

Now do:

## Weighing 2
Weigh:

**1 2 5** vs **3 6 9**

Here **9** is known genuine from the first weighing if needed only when balanced?  
Better to use a coin known genuine only if we have one. Since we don’t yet know one in this branch, use a carefully designed standard approach:

### Instead, do this:
Weigh:

**1 2 5** vs **3 6 9**

This requires coin 9 to be genuine, so this branch is not ideal unless we first establish a genuine coin.  
So here is the clean, correct standard decision tree:

---

## A fully correct strategy

### Weighing 1
**1 2 3 4** vs **5 6 7 8**

---

## If it balances
Then counterfeit is among **9,10,11,12** and proceed as in Case 1.

---

## If left side is heavier
Then possible candidates are:

- **1,2,3,4** are **heavy**
- **5,6,7,8** are **light**

### Weighing 2
Weigh:

**1 2 5** vs **3 6 9**

Now interpret:

- If **left heavier**, then either:
  - **1 heavy**, or
  - **2 heavy**, or
  - **6 light**
- If **right heavier**, then either:
  - **3 heavy**, or
  - **5 light**
- If **balances**, then either:
  - **4 heavy**, or
  - **7 light**, or
  - **8 light**

To finish:

### Weighing 3
- If candidates are **1,2,6**:
  weigh **1 vs 2**
  - if one is heavier, that’s the counterfeit
  - if they balance, **6 is light**
- If candidates are **3,5**:
  weigh **3 vs 9**
  - if 3 is heavier, **3 is counterfeit heavy**
  - otherwise **5 is light**
- If candidates are **4,7,8**:
  weigh **7 vs 8**
  - if one is lighter, that’s the counterfeit
  - if they balance, **4 is heavy**

---

## If right side is heavier
Then the situation is symmetric:

Possible candidates are:

- **5,6,7,8** are **heavy**
- **1,2,3,4** are **light**

### Weighing 2
Weigh:

**5 6 1** vs **7 2 9**

Interpret similarly:

- If left heavier: candidates among **5 heavy, 6 heavy, 2 light**
- If right heavier: candidates among **7 heavy, 1 light**
- If balances: candidates among **8 heavy, 3 light, 4 light**

Then use Weighing 3 to distinguish exactly as above.

---

# A cleaner compact version
A very common way to present this is as a **decision tree**:

1. **Weigh 1–4 vs 5–8**
2. Based on the result, choose a second weighing that splits the remaining possibilities into at most 3 cases
3. Use the third weighing to resolve the final 2–3 suspects

This works because each weighing has 3 outcomes, and \(3^3 = 27\), enough to distinguish the \(12 \times 2 = 24\) possibilities.

---

If you want, I can give you a **fully tabulated, exact step-by-step chart** for all 27 outcomes so you can follow it mechanically without thinking.

## 4. Evaluating the response

I then pass both the question and answer to another LLM call and ask it to judge whether the answer is correct.

This gives the workflow a simple **generate → answer → evaluate** pattern.


In [16]:
# Combine the question and answer into an evaluation prompt.

message = f"""
Here is a question:
{question}

And here is a possible answer that might be correct or incorrect:
{answer}

Please evaluate if the answer is correct or incorrect.
"""

print(message)



Here is a question:
You have 12 identical-looking coins, one of which is counterfeit and may be either heavier or lighter than the others. Using a balance scale exactly three times, what is a strategy that will always identify the counterfeit coin and determine whether it is heavier or lighter?

And here is a possible answer that might be correct or incorrect:
Here’s a standard 3-weighing strategy that always works for **12 coins**, with **one counterfeit coin** that could be **heavier or lighter**.

I’ll label the coins:

**1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12**

---

## Weighing 1
Weigh:

**1 2 3 4** vs **5 6 7 8**

There are three possible outcomes.

---

# Case 1: The first weighing balances
Then coins **1–8 are genuine**, so the counterfeit is among **9, 10, 11, 12**.

## Weighing 2
Weigh:

**9 10 11** vs **1 2 3**

Since 1, 2, 3 are genuine:

- If left side is heavier, then one of **9, 10, 11** is **heavy**
- If left side is lighter, then one of **9, 10, 11** is **light**
- If 

In [17]:
# Ask another model to evaluate the answer.

messages = [{"role": "user", "content": message}]
response = openai.chat.completions.create(model="gpt-5.4", messages=messages)
print(response.choices[0].message.content)


The answer is **incorrect**.

Why:

- The **Case 1** branch, where the first weighing balances, is fine.
- But in the **unbalanced** branch, the proposed second weighing uses **coin 9 as a known genuine coin**, which is actually valid here, because if coins **1–8** contain the counterfeit, then **9–12 are genuine**. So that part is not the issue.

The real problem is that the interpretation of the second weighing is **wrong/incomplete**.

Example:

After first weighing:

- **1 2 3 4 > 5 6 7 8**

So possibilities are:
- **1,2,3,4 heavy**
- **5,6,7,8 light**

Second weighing proposed:
- **1 2 5** vs **3 6 9**

Now test possibility **5 is light**:
- Left pan has **1,2,5**
- Right pan has **3,6,9**
- Since **5 is light**, the **left pan becomes lighter**, so the **right pan is heavier**

So **5 light** belongs in the “right heavier” outcome.

Test possibility **6 is light**:
- Right pan contains **6**
- If **6 is light**, the **right pan becomes lighter**, so the **left pan is heavier**

S

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try this commercial application:<br/>
            First ask the LLM to pick a business area that might be worth exploring for an Agentic AI opportunity.<br/>
            Then ask the LLM to present a pain-point in that industry - something challenging that might be ripe for an Agentic solution.<br/>
            Finally have 3 third LLM call propose the Agentic AI solution. <br/>
            We will cover this at up-coming labs, so don't worry if you're unsure.. just give it a try!
            </span>
        </td>
    </tr>
</table>

## 5. My agriculture experiment

I applied the same idea to an agriculture use case: identify a pain point and ask the model to propose an Agentic AI solution.

This helped me connect the technical exercise to a domain I already understand.


In [18]:
# Identify an agriculture pain point that could be explored by an AI agent.


messages = [{"role": "user", "content": "what is one pain-point in agriculture that is worth exploring by an agent. In plain text and just the pain-point"}]


response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
question = response.choices[0].message.content
print(question)


Unpredictable pest and disease outbreaks causing sudden crop losses


In [19]:
# Turn the identified pain point into a proposed Agentic AI solution.

messages = [{
    "role": "user",
    "content": f"""
Here is a pain-point in agriculture:
{question}

Propose an Agentic AI solution for this pain-point.
Briefly explain what the agent would do, which tools it might use, and how it would help.
Respond in plain text only.
Do not use markdown.
"""
}]

response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
solution = response.choices[0].message.content
print(solution)



An agentic AI solution could be a Crop Health Monitoring Agent that continuously watches for early signs of pest and disease pressure, predicts outbreak risk, and recommends timely actions before losses spread.

What the agent would do:
It would combine data from field sensors, drone or satellite imagery, weather forecasts, crop stage, and historical farm records to detect unusual patterns such as leaf discoloration, canopy stress, pest hotspots, or conditions favorable for disease. When risk rises, it would alert the farmer, explain the likely threat, and suggest the best response, such as scouting a specific area, applying biological control, adjusting irrigation, or scheduling a targeted spray.

Tools it might use:
It could use computer vision models for image analysis, weather and disease-risk models, GIS mapping tools, pest and disease databases, farm management software, and mobile messaging or voice assistants to communicate with workers. If connected to drones or field cameras,

## Key Takeaway

This lab introduced me to the idea of connecting multiple LLM calls into a workflow.

I also started experimenting with how these patterns could be applied to agriculture, something I’ll build on in later labs.
